<a href="https://colab.research.google.com/github/alaaguedda/medical_report_summarization_project/blob/enhanced/medical_summerizer_trained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d aminexdr/bhc-mimic-iv-summary
!unzip bhc-mimic-iv-summary.zip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/aminexdr/bhc-mimic-iv-summary
License(s): unknown
 92% 409M/446M [00:00<00:00, 475MB/s]
100% 446M/446M [00:00<00:00, 518MB/s]
Archive:  bhc-mimic-iv-summary.zip
  inflating: BHC_MIMIC-IV.csv        


In [4]:
import pandas as pd

df = pd.read_csv("BHC_MIMIC-IV.csv")


In [5]:
import pandas as pd

# 1. Basic Cleaning & Sampling
df = df[['input', 'target']].dropna()
df = df.sample(frac=1, random_state=42).iloc[:12000]

# 2. Helper function to slice by word count
def limit_words(text, limit):
    if not isinstance(text, str):
        return ""
    words = text.split()
    return " ".join(words[:limit])

# 3. Apply the limits
df['input'] = df['input'].apply(lambda x: limit_words(x, 300))   # more context
df['target'] = df['target'].apply(lambda x: limit_words(x, 80))  # still compressed

print(f"Dataframe processed. Shape: {df.shape}")

Dataframe processed. Shape: (12000, 2)


In [6]:
df.head()

,input,target
267505,write a discharge summary: History of Present ...,Patient arrived on the unit intubated and seda...
180880,generate a brief hospital summary: Chief Compl...,"AP: year old female with ho CLL, PAF, not on c..."
252848,create a summary based on the following inform...,Mrs. is a yr old female presenting with new ab...
112160,write a discharge summary: Chief Complaint: Ch...,Patient had recurring chest pain consistent wi...
99808,create a summary based on the following inform...,"Mr. is a M w ho CAD , atrial fibrillation sp c..."


In [8]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path where you want to save it
# Change 'my_folder' to whatever folder exists in your Drive
save_path = '/content/drive/MyDrive/medical_report.csv'

# Save the dataframe
df.to_csv(save_path, index=False)

print(f"File successfully saved to: {save_path}")

Mounted at /content/drive
File successfully saved to: /content/drive/MyDrive/medical_report.csv


In [1]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/medical_report.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df.head(
)

In [ ]:
df.isnull().sum()
df = df.dropna()

In [3]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [4]:
!pip install transformers datasets evaluate rouge-score

In [5]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
def preprocess_function(examples):
    # Guarantee the prefix is always present and consistent
    inputs = [
        "summarize: " + str(text) if not str(text).startswith("summarize:") else str(text)
        for text in examples["input"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True,
        padding=False
    )

    # ✅ Just tokenize labels directly — as_target_tokenizer() is deprecated/removed
    labels = tokenizer(
        text_target=examples["target"],   # use text_target= instead
        max_length=64,
        truncation=True,
        padding=False
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [7]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8   # helps with fp16 performance on T4
)

In [8]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/9600 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [9]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    eval_accumulation_steps=10,
    gradient_accumulation_steps=2,
    load_best_model_at_end=True,
    save_strategy="epoch",          # needed for early stopping
    metric_for_best_model="eval_ROUGE-L",   # ✅ add eval_ prefix explicitly
    greater_is_better=True,
    num_train_epochs=6,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,  # Now this will work!
    generation_max_length=64,
    generation_num_beams=4,
    logging_steps=200,
    report_to="none"
)

In [10]:
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Fix: clip predictions to valid token id range
    vocab_size = tokenizer.vocab_size
    predictions = np.clip(predictions, 0, vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Fix: also clip labels just in case
    labels = np.clip(labels, 0, vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "ROUGE-1": rouge_result["rouge1"],
        "ROUGE-2": rouge_result["rouge2"],
        "ROUGE-L": rouge_result["rougeL"],
    }

In [11]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,                          # ✅ must be here
)

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge-1,Rouge-2,Rouge-l
1,5.381284,2.430978,0.381162,0.226717,0.333033
2,4.965800,2.297072,0.385556,0.230850,0.335772
3,4.614743,2.229650,0.389977,0.234556,0.339840
4,4.523457,2.187031,0.390658,0.235033,0.339674
5,4.418129,2.158749,0.391650,0.234508,0.340647
6,4.333705,2.150757,0.394322,0.238891,0.343557


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3600, training_loss=4.796811150444879, metrics={'train_runtime': 4340.4654, 'train_samples_per_second': 13.27, 'train_steps_per_second': 0.829, 'total_flos': 3897843882393600.0, 'train_loss': 4.796811150444879, 'epoch': 6.0})

In [13]:
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/medical_summarizer_t5small"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /content/drive/MyDrive/medical_summarizer_t5small


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import T5Tokenizer, T5ForConditionalGeneration

load_path = "/content/drive/MyDrive/medical_summarizer_t5small"

tokenizer = T5Tokenizer.from_pretrained(load_path)
model = T5ForConditionalGeneration.from_pretrained(load_path)
model.eval()

print("✅ Model loaded and ready!")

In [14]:
!pip install evaluate rouge_score absl-py

In [ ]:
import torch
import gc

# 1. Clear Python garbage collector
gc.collect()

# 2. Clear NVIDIA cache
torch.cuda.empty_cache()



In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,  # Fix: renamed from 'tokenizer'
    compute_metrics=compute_metrics
)

In [15]:
results = trainer.evaluate()

In [16]:
print(results)

{'eval_loss': 2.150757312774658, 'eval_ROUGE-1': 0.3943221813977008, 'eval_ROUGE-2': 0.2388911301342408, 'eval_ROUGE-L': 0.3435565282219455, 'eval_runtime': 623.3179, 'eval_samples_per_second': 1.925, 'eval_steps_per_second': 0.963, 'epoch': 6.0}


In [29]:
import torch
import textwrap

def generate_and_compare(sample_input, reference_summary=None,
                         input_preview_chars=500,
                         max_input_length=512,
                         max_target_length=90,
                         num_beams=4):
    """
    Generates summary and prints structured comparison.

    Parameters
    ----------
    sample_input : str
        The original medical report text.
    reference_summary : str, optional
        The ground truth summary (if available).
    input_preview_chars : int
        Number of characters from original input to display.
    max_input_length : int
        Token truncation length for input.
    max_target_length : int
        Maximum generation length.
    num_beams : int
        Beam search width.
    """

    device = model.device
    model.eval()

    # ---- Prepare Input ----
    input_text = "summarize: " + sample_input

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # ---- Generate ----
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_target_length,
            num_beams=num_beams,
            early_stopping=True
        )

    generated_summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # ---- Pretty Printing Section ----
    print("\n" + "="*80)
    print(" ORIGINAL INPUT (Preview)")
    print("="*80)
    print(textwrap.fill(sample_input[:input_preview_chars], width=100))

    print("\n" + "-"*80)
    print(" GENERATED SUMMARY")
    print("-"*80)
    print(textwrap.fill(generated_summary, width=100))


    print("="*80 + "\n")

    return generated_summary

In [ ]:
generate_and_compare(
    sample_input=df['input'].iloc[4],
)

In [33]:
test_report = """
	write a discharge summary: History of Present Illness: The patients oncologic history began in , at
which time he felt a pain in his left side that was initially attributed to musculoskeletal strain.
However, this pain did not remit with over-the-counter analgesics, and a physician in an abdominal
CT scan. According to the patient and his wife , this revealed lymphadenopathy suspicious for
lymphoma. He then underwent upper endoscopy with biopsy of a gastric lesion in approximately , which
report"""

generate_and_compare(sample_input=test_report)


📝 ORIGINAL INPUT (Preview)
         write a discharge summary: History of Present Illness: The patients oncologic history began
in , at which time he felt a pain in his left side that was initially attributed to musculoskeletal
strain. However, this pain did not remit with over-the-counter analgesics, and a physician in an
abdominal CT scan. According to the patient and his wife , this revealed lymphadenopathy suspicious
for lymphoma. He then underwent upper endoscopy with biopsy of a gastric lesion in approximately ,
which repo

--------------------------------------------------------------------------------
🤖 GENERATED SUMMARY
--------------------------------------------------------------------------------
The patient was admitted to the plastic surgery service on and had a gastric lesion biopsy. The
patient tolerated the procedure well and returned to the PACU in stable condition.



'The patient was admitted to the plastic surgery service on and had a gastric lesion biopsy. The patient tolerated the procedure well and returned to the PACU in stable condition.'